**🧹 What is the VACUUM Command?**
- The VACUUM command is used to clean up old files that are no longer needed in a Delta table.
- Think of it like emptying the trash bin on your computer to free up space.

**🧠 Why is it needed?**
- Delta Lake keeps old versions of data so you can use features like RESTORE, TIME TRAVEL, and DESCRIBE HISTORY.
- But over time, these old files take up storage space. If you don’t need them anymore, you can run VACUUM to delete them.

**✅ Simple Example**
- This will remove files that are older than 7 days (default setting).
- from delta.tables import DeltaTable
- deltaTable = DeltaTable.forPath(spark, "/mnt/delta/sales_data")
- deltaTable.vacuum()

**⏱️ Custom Retention Period**
- You can change how old the files must be before they’re deleted:
- This keeps files from the last 7 days (168 hours) and deletes older ones.
- VACUUM sales_data RETAIN 168 HOURS;

________________________________________
**⚠️ Important Notes**
- •	Time travel will not work for deleted versions after vacuuming.
- •	You can’t vacuum files newer than 7 days unless you disable safety checks.
- •	Be careful in production—don’t vacuum too aggressively!
________________________________________
**🔒 Safety Check**
Delta Lake has a safety feature to prevent accidental data loss. If you try to vacuum files newer than 7 days, you’ll get an error unless you disable the check:

**Then you can run:**
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

**Then you can run:**
VACUUM sales_data RETAIN 1 HOURS;


In [0]:
from delta.tables import *
DeltaTable.createOrReplace(spark) \
    .tableName("tri.tri.scd2Demo5") \
    .addColumn("pk1", "INT") \
    .addColumn("pk2", "STRING") \
    .addColumn("dim1", "INT") \
    .addColumn("dim2", "INT") \
    .addColumn("dim3", "INT") \
    .addColumn("dim4", "INT") \
    .addColumn("active_status", "STRING") \
    .addColumn("start_date", "TIMESTAMP") \
    .addColumn("end_date", "TIMESTAMP") \
    .execute()

In [0]:
%sql
insert into tri.tri.scd2Demo5 values (111,'Unit1',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo5 values (222,'Unit2',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo5 values (333,'Unit3',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

In [0]:
%sql
insert into tri.tri.scd2Demo5 values (666,'Unit6',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo5 values (777,'Unit7',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo5 values (888,'Unit8',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

### Here we store the table into a particular location so we can perform operations on top of the path or location

In [0]:
spark.table("tri.tri.scd2Demo5") \
    .write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/tri/tri/tri_files/scd2Demo5")


In [0]:
%sql
SELECT * FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo5`

### create table instance

In [0]:
from delta.tables import DeltaTable
target_table=DeltaTable.forPath(spark, "/Volumes/tri/tri/tri_files/scd2Demo5")
display(target_table.toDF())

In [0]:
display(target_table.toDF().where("pk1 = 111"))




In [0]:
target_table.update(
    condition="pk1 == 333",
    set={"dim2": "5001"}
)

In [0]:
display(target_table.history())

### apply z order

In [0]:
target_table.optimize().executeZOrderBy("PK1")


In [0]:
display(target_table.optimize().executeZOrderBy("PK1"))

### Check if OPTIMIZE is applying automatically
If you think OPTIMIZE is applying automatically, you can check using the process below:

### verify

In [0]:
target_table.history().select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)

### Cluster vs Serverless for Auto Compaction & Optimized Writes

If you are using a **cluster** (classic compute), you can check and set **Auto Compaction** and **Optimized Writes** to `true` or `false` using `spark.conf`.

However, on **Serverless** compute, these settings are managed automatically and **cannot** be manually checked or overridden.


**delta.autoOptimize.autoCompact**

Purpose: Automatically combines small files into larger files after INSERT, UPDATE, MERGE, etc.

### Check Auto Compaction true or false

If the result is **true**, Auto Compaction is **enabled**.
If the result is **false** (or the default is returned), Auto Compaction is **disabled**.
To enable it, set the property to `true`; to disable it, set it to `false`.

In [0]:
spark.conf.get(
    "spark.databricks.delta.autoCompact.enabled",
    "false"
)

### Example: if I get `true`, then disable it using the below query

### Disable Auto Compaction

In [0]:
spark.conf.set(
    "spark.databricks.delta.autoCompact.enabled",
    "false"
)

%md
**delta.autoOptimize.optimizeWrite**

Purpose: Writes data into larger, better-sized files during the write operation itself, reducing small-file creation.

### Check Optimized Writes

In [0]:
spark.conf.get(
    "spark.databricks.delta.optimizeWrite.enabled",
    "false"
)

### Disable Optimized `Writes`

In [0]:
spark.conf.set(
    "spark.databricks.delta.optimizeWrite.enabled",
    "false"
)

### Verify sql

In [0]:
%sql
SHOW TBLPROPERTIES delta.`/Volumes/tri/tri/tri_files/scd2Demo5`;

### Check Auto Compaction (SQL)

In [0]:
%sql
SHOW TBLPROPERTIES tri.tri.scd2Demo5 (
  delta.autoOptimize.autoCompact
);

### Disable Auto Compaction

In [0]:
%sql
ALTER TABLE tri.tri.scd2Demo3
SET TBLPROPERTIES (
  'delta.autoOptimize.autoCompact' = 'false'
);

### Check Optimize Write (SQL)

In [0]:
%sql
SHOW TBLPROPERTIES tri.tri.scd2Demo5(
  delta.autoOptimize.optimizeWrite
);

### Disable Optimize Write

In [0]:
%sql
ALTER TABLE tri.tri.scd2Demo3
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false'
);

if i set table properties it will show the true or false value

### performing vaccume

In [0]:
display(dbutils.fs.ls("/Volumes/tri/tri/tri_files/scd2Demo5"))

In [0]:
target_table.history().display()


**DeltaTable instance supports methods such as:**

- target_table.update()
- target_table.delete()
- target_table.merge()
- target_table.history()
- target_table.optimize()

### top of table spark quary
### VACUUM (Default Retention)

In [0]:
spark.sql("""
VACUUM tri.tri.scd2Demo5
""")

### top of path spark quary
### VACUUM (Default Retention)

In [0]:
spark.sql("""
VACUUM delta.`/Volumes/tri/tri/tri_files/scd2Demo5`
""")

### top of table spark quary
### 
###  VACUUM DRY RUN

In [0]:
spark.sql("""
VACUUM tri.tri.scd2Demo5 DRY RUN
""")

### top of path spark quary
### 
### VACUUM DRY RUN

In [0]:
spark.sql("""
VACUUM delta.`/Volumes/tri/tri/tri_files/scd2Demo5`
DRY RUN
""").show(truncate=False)

### VACUUM with Custom Retention on top  of table

In [0]:
spark.sql("""
VACUUM tri.tri.scd2Demo5 RETAIN 0 HOURS dry run
""")

### VACUUM with Custom Retention on top of path

In [0]:
spark.sql("""
VACUUM delta.`/Volumes/tri/tri/tri_files/scd2Demo5`
RETAIN 168 HOURS
""")

it will throw error if you try to delete the table

### Purpose: Disables the Delta Lake safety check that prevents VACUUM from using very small retention periods.

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

now you can apply this 

spark.sql("""
VACUUM tri.tri.scd2Demo5 RETAIN 0 HOURS dry run
""")

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/tri/tri/tri_files/scd2Demo5`;

### Check Specific Retention Property

In [0]:
%sql
SHOW TBLPROPERTIES delta.`/Volumes/tri/tri/tri_files/scd2Demo5`
(
  delta.deletedFileRetentionDuration
);

In [0]:
spark.sql("""
SHOW TBLPROPERTIES delta.`/Volumes/tri/tri/tri_files/scd2Demo5`
(
  delta.deletedFileRetentionDuration
)
""").show(truncate=False)

**output**

if property set

- key                                       value
- delta.deletedFileRetentionDuration       interval 7 days

**output**

If the property is not set:

- Table ... does not have property:
- delta.deletedFileRetentionDuration